In [28]:
# -*- coding: utf-8 -*-
"""
Incendis forestals a Catalunya (1998-2024) — Visualitzacions Finals Q1-Q4
Information Visualization | Projecte 1
"""

import copy
import json
import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

DATA_PATH = "data/incendis_final.csv"
COMARQUES_MAP_PATH = 'data/divisions-administratives-v2r2-comarques-500000-20260120.json'

df = pd.read_csv(DATA_PATH)
top_causes = df['Causa'].value_counts().nlargest(6).index
df['CausaAgrupada'] = df['Causa'].where(df['Causa'].isin(top_causes), 'Altres')
causa_domain = sorted(df['CausaAgrupada'].unique())
causa_scale = alt.Scale(domain=causa_domain, scheme='dark2')


# --- GeoJSON i càlcul d'aspect ratio real ---
geojson_path = (
    COMARQUES_MAP_PATH
)
with open(geojson_path, encoding='utf-8') as f:
    geo_dict = json.load(f)
comarques_geo = alt.Data(values=geo_dict['features'])
noms_geojson = {feat['properties']['NOMCOMAR'] for feat in geo_dict['features']}


def _calcular_bbox(geo_d):
    xs, ys = [], []

    def recorre(coords):
        if isinstance(coords[0], (int, float)):
            xs.append(coords[0])
            ys.append(coords[1])
        else:
            for c in coords:
                recorre(c)

    for feat in geo_d['features']:
        recorre(feat['geometry']['coordinates'])
    return min(xs), max(xs), min(ys), max(ys)


_minx, _maxx, _miny, _maxy = _calcular_bbox(geo_dict)
ASPECT_RATIO = (_maxx - _minx) / (_maxy - _miny)


def mida_mapa(amplada):
    return amplada, round(amplada / ASPECT_RATIO)


PROJECCIO = 'identity'
REFLECTEIX_Y = True


def titol_rt(titol, subtitol):
    return alt.TitleParams(
        text=titol,
        subtitle=subtitol,
        anchor='start',
        fontSize=20,
        fontWeight='bold',
        subtitleFontSize=13,
        subtitleColor='#555555',
        offset=12
    )

In [ ]:
import altair as alt
import numpy as np
import pandas as pd

COLORS = ["#56B4E9", "#E69F00", "#009E73", "#D55E00", "#999999", "#CC79A7"]

LEGEND = alt.Legend(title="Cause of fire")


def _cause_order(d: pd.DataFrame, cause_col: str = "CausaAgrupada") -> list[str]:
    return d[cause_col].value_counts().index.tolist()


def _cause_scale(causes: list[str], palette: list[str] = COLORS) -> alt.Scale:
    return alt.Scale(
        domain=causes,
        range=[palette[i % len(palette)] for i in range(len(causes))],
    )


# ------------------- 6. wide beeswarm: one dot = `per_dot` fires -------------
def beeswarm_data(
    d: pd.DataFrame,
    per_dot: int = 5,
    cols: int = 7,
    spread: float = 0.9,
    layout: str = "stacked",
    cause_col: str = "CausaAgrupada",
) -> pd.DataFrame:
    """One row per dot (1 dot = `per_dot` fires; cumulative rounding per year).
    layout="lanes"   : one horizontal lane per cause in `cause_col`; dots form a swarm
                       centred on the lane -> every cause shares a baseline.
    layout="stacked" : dots of all causes packed bottom-up inside each year.
    """
    causes = _cause_order(d, cause_col)
    dx = spread / cols

    counts_all = (
        d.groupby(["yr", cause_col])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=causes, fill_value=0)
        .sort_index()
    )

    dots = {}
    for yr, counts in counts_all.iterrows():
        cum, prev = counts.cumsum(), 0
        for cause in causes:
            end = int(round(cum[cause] / per_dot))
            dots[(yr, cause)] = (end - prev, int(counts[cause]), prev)
            prev = end

    lane_rows = max((-(-nd // cols) for nd, _, _ in dots.values()), default=1)
    lane_h = lane_rows + 2

    out = []
    for (yr, cause), (nd, fires, first) in dots.items():
        nrows = -(-nd // cols) if nd else 0
        for j in range(nd):
            k = first + j if layout == "stacked" else j
            r, c = divmod(k, cols)
            x = yr + (c - (cols - 1) / 2) * dx + (dx / 4 if r % 2 == 0 else -dx / 4)
            if layout == "lanes":
                y = (len(causes) - 1 - causes.index(cause)) * lane_h + (r - (nrows - 1) / 2) * 0.866 + lane_h / 2
            else:
                y = r * 0.866
            out.append({"yr": int(yr), cause_col: cause, "x": x, "y": y, "fires": fires})

    bees = pd.DataFrame(out)
    bees.attrs["lane_h"] = lane_h
    bees.attrs["causes"] = causes
    bees.attrs["cause_col"] = cause_col
    return bees


def chart6(
    d: pd.DataFrame,
    per_dot: int = 5,
    cols: int = 7,
    width: int = 1500,
    layout: str = "stacked",
    cause_col: str = "CausaAgrupada",
):
    """Wide beeswarm of fires per year, coloured by `CausaAgrupada`. 1 dot = `per_dot` fires.
    layout="stacked" (default): all causes stacked in one swarm per year.
    layout="lanes": one lane per cause.
    """
    b = beeswarm_data(d, per_dot, cols, layout=layout, cause_col=cause_col)
    causes = b.attrs["causes"]
    cause_scale = _cause_scale(causes)
    lane_h = b.attrs["lane_h"]

    yr_min = int(d["yr"].min()) if not d.empty else 1998
    yr_max = int(d["yr"].max()) if not d.empty else 2024
    n_years = max(yr_max - yr_min + 1, 1)

    px_year = width / n_years
    dx_px = px_year * 0.9 / cols
    ymax = (len(causes) * lane_h) if layout == "lanes" else (float(b["y"].max()) + 1 if not b.empty else 1.0)
    height = max(int(ymax * dx_px) + 30, 120)
    size = (dx_px * 0.92) ** 2

    xenc = alt.X(
        "x:Q",
        title="Year",
        scale=alt.Scale(domain=[yr_min - 0.5, yr_max + 0.5], nice=False, zero=False),
        axis=alt.Axis(
            format="d",
            values=list(range(yr_min, yr_max + 1)),
            grid=False,
            labelAngle=0,
            labelFontSize=11,
        ),
    )
    title = f"Fires per year by cause (1 dot = {per_dot} fires)"

    
    step_fires = 100 if per_dot <= 5 else 200
    max_fires_approx = int(np.ceil(ymax / 0.866 * cols * per_dot))
    yenc = alt.Y(
        "y:Q",
        title="Fires per year (approx.)",
        scale=alt.Scale(domain=[-1, ymax], nice=False),
        axis=alt.Axis(
            grid=True,
            gridDash=[2, 3],
            labelExpr=f"round(datum.value / 0.866 * {cols * per_dot})",
            values=[0.866 * v / (cols * per_dot) for v in range(0, max_fires_approx + 1, step_fires)],
        ),
    )
    legend = LEGEND
    bands = None

    dots = (
        alt.Chart(b)
        .mark_circle(size=size, opacity=1, stroke="white", strokeWidth=0.4)
        .encode(
            x=xenc,
            y=yenc,
            color=alt.Color(f"{cause_col}:N", scale=cause_scale, legend=legend),
            tooltip=[
                alt.Tooltip("yr:O", title="Year"),
                alt.Tooltip(f"{cause_col}:N", title="Cause"),
                alt.Tooltip("fires:Q", title="Fires of this cause that year"),
            ],
        )
    )

    chart = bands + dots if bands is not None else dots
    return chart.properties(width=width, height=height, title=title)



In [37]:
"""Q2 - Evolution of the causes of fires: five Altair proposals.

Input: the cleaned fires table (OpenRefine export) with at least the columns
    Campania, Origen, Causa, SuperficieTotalForestal
Usage in Colab / Streamlit:
    from q2_charts import prepare, chart1, chart2, chart3, chart4, chart5
    d = prepare(df)            # df = your cleaned DataFrame
    chart1(d)                  # etc.
"""
import altair as alt
import pandas as pd


COLORS = ["#56B4E9", "#E69F00", "#009E73", "#D55E00", "#999999", "#CC79A7"]

LEGEND = alt.Legend(title="Cause of fire")
ORDRE_CAUSA = df["CausaAgrupada"].value_counts().index.tolist()
ESCALA_CAUSA = alt.Scale(domain=ORDRE_CAUSA, range=[COLORS[i % len(COLORS)] for i in range(len(ORDRE_CAUSA))])


# ------------------- 6. wide beeswarm: one dot = `per_dot` fires -------------
def beeswarm_data(d, per_dot=5, cols=7, spread=0.9, layout="stacked"):
    """One row per dot (1 dot = `per_dot` fires; cumulative rounding per year).
    layout="lanes"   : one horizontal lane per cause family; dots form a swarm
                       centred on the lane -> every cause shares a baseline.
    layout="stacked" : dots of all causes packed bottom-up inside each year."""
    dx = spread / cols
    counts_all = d.groupby(["Any", "CausaAgrupada"]).size().unstack(fill_value=0)
    causes = ORDRE_CAUSA
    #Altair no pot fer beeswarms directament osigui que hem de generar els dots nosaltres
    dots = {}

    for yr, counts in counts_all.iterrows():
        cum, prev = counts.cumsum(), 0
        for causa in causes:
            end = int(round(cum[causa] / per_dot))
            dots[(yr, causa)] = (end - prev, int(counts[causa]), prev)   # n dots, fires, first dot index
            prev = end

    lane_rows = max(-(-nd // cols) for nd, _, _ in dots.values())   # tallest swarm, in rows
    lane_h = lane_rows + 2

    out = []
    for (yr, causa), (nd, fires, first) in dots.items():
        nrows = -(-nd // cols) if nd else 0
        for j in range(nd):
            k = first + j if layout == "stacked" else j
            r, c = divmod(k, cols)
            x = yr + (c - (cols - 1) / 2) * dx + (dx / 4 if r % 2 == 0 else -dx / 4)
            if layout == "lanes":
                y = (len(causes) - 1 - causes.index(causa)) * lane_h + (r - (nrows - 1) / 2) * 0.866 + lane_h / 2
            else:
                y = r * 0.866
            out.append({"Any""": yr, "family": causa, "x": x, "y": y, "fires": fires})
    df = pd.DataFrame(out)
    df.attrs["lane_h"] = lane_h
    df.attrs["causes"] = causes
    
    return df


def chart6(d, per_dot=5, cols=7, width=1500, layout="stacked"):
    """Wide beeswarm of fires per year, coloured by cause. 1 dot = `per_dot` fires.
    layout="stacked" (default): all causes stacked in one swarm per year.
    layout="lanes": one lane per cause."""
    b = beeswarm_data(d, per_dot, cols, layout=layout)
    lane_h = b.attrs["lane_h"]
    causes = b.attrs["causes"]
    cause_scale = ESCALA_CAUSA

    yr_min = int(d["Any"].min())
    yr_max = int(d["Any"].max())
    n_years = max(yr_max - yr_min + 1, 1)

    px_year = width / n_years
    dx_px = px_year * 0.9 / cols                      # pixel distance between dots
    ymax = float(b["y"].max()) + 1 if not b.empty else 1.0
    height = max(int(ymax * dx_px) + 30, 120)
    size = (dx_px * 0.92) ** 2     

    xenc = alt.X("x:Q", title="Year",
                 scale=alt.Scale(domain=[yr_min - 0.5, yr_max + 0.5], nice=False, zero=False),
                 axis=alt.Axis(format="d", values=list(range(yr_min, yr_max + 1)), 
                                grid=False, labelFontSize=11))
    title = f"Fires per year by cause (1 dot = {per_dot} fires)"
    
    
    yenc = alt.Y("y:Q", title="Fires per year (approx.)", scale=alt.Scale(domain=[-1, ymax], nice=False),
                    axis=alt.Axis(grid=True, gridDash=[2, 3],
                                labelExpr=f"round(datum.value / 0.866 * {cols * per_dot})",
                                values=[0.866 * v / (cols * per_dot) * 1 for v in range(0, int(ymax / 0.866 * cols * per_dot) + 1, 100 if per_dot <= 5 else 200)]))
    legend = LEGEND


    dots = (
    alt.Chart(b)
    .mark_circle(size=size, opacity=1, stroke="white", strokeWidth=0.4)
    .encode(x=xenc, y=yenc,
            color=alt.Color("family:N", scale=ESCALA_CAUSA, legend=legend),
            tooltip=["yr:O", "family:N", alt.Tooltip("fires:Q", title="Fires of this cause that year")])
    )
    chart = dots
    return chart.properties(width=width, height=height, title=title)

q2 = chart6(df)
q2.save("outputs/Q2.json")

